# Step 6 — Model Comparison

Compares ESAS PANNs CNN14 against YAMNet, AST, and CLAP on the same ESC-50 hazard-tier splits.

All models use the same evaluation protocol:
- Same 3-tier taxonomy (HIGH / MEDIUM / LOW)
- Same official ESC-50 5-fold CV
- Logistic regression head on frozen embeddings
- Macro F1 as primary metric

**Install requirements first:**
```bash
pip install transformers torchaudio timm laion-clap
```

**Expected time:** ~2-3 hours on CPU (embeddings are cached after first run)

In [ ]:
pip install transformers torchaudio timm laion-clap

In [ ]:
from pathlib import Path

# ── Set your base path ─────────────────────────────────────────────────────
BASE = Path('/path/to/your/esas_project')  # <-- set this
# ───────────────────────────────────────────────────────────────────────────

ESC50_DIR   = BASE / 'ESC-50'
MODELS_DIR  = Path('models')
RESULTS_DIR = Path('results')
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

print(f'ESC-50: {ESC50_DIR.exists()}')

In [ ]:
import csv, glob, json, warnings
from collections import Counter
import numpy as np
import librosa
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_recall_fscore_support
warnings.filterwarnings('ignore')

HIGH   = {'car_horn','chainsaw','crackling_fire','glass_breaking','hand_saw','siren','fireworks'}
MEDIUM = {'clock_alarm','clock_tick','crying_baby','dog','door_wood_knock','sneezing',
           'coughing','church_bells','vacuum_cleaner','washing_machine','toilet_flush','cat'}

def get_priority(label):
    if label in HIGH:   return 'HIGH'
    if label in MEDIUM: return 'MEDIUM'
    return 'LOW'

# Load metadata
meta = {}
with open(ESC50_DIR / 'meta' / 'esc50.csv') as f:
    for row in csv.DictReader(f):
        meta[row['filename']] = {'cat': row['category'], 'fold': int(row['fold'])}

files = sorted(glob.glob(str(ESC50_DIR / 'audio' / '*.wav')))
print(f'ESC-50: {len(files)} clips, {len(meta)} in metadata')

def cross_validate(X, y, folds):
    le = LabelEncoder()
    ye = le.fit_transform(y)
    all_true, all_pred = [], []
    for fold in range(1, 6):
        tr = folds != fold; te = folds == fold
        clf = LogisticRegression(C=1.0, max_iter=1000, random_state=42, class_weight='balanced')
        clf.fit(X[tr], ye[tr])
        all_pred.extend(clf.predict(X[te]))
        all_true.extend(ye[te])
        print(f'  Fold {fold}/5 done')
    all_true = np.array(all_true); all_pred = np.array(all_pred)
    _, _, f, _ = precision_recall_fscore_support(all_true, all_pred,
                     labels=range(len(le.classes_)), zero_division=0)
    _, _, mf, _ = precision_recall_fscore_support(all_true, all_pred,
                      average='macro', zero_division=0)
    return {c: round(float(f[i]),3) for i,c in enumerate(le.classes_)}, round(float(mf),3)

def load_audio(path, sr=16000):
    audio, _ = librosa.load(path, sr=sr, mono=True)
    if len(audio) < sr: audio = np.pad(audio, (0, sr-len(audio)))
    else: audio = audio[:sr*10]  # max 10s
    return audio

all_results = {}
print('Setup done')

## Model 1 — PANNs CNN14 (ESAS baseline, already in results)

In [ ]:
# Load cached results from notebook 02
esc_path = RESULTS_DIR / 'esc50_results.json'
if esc_path.exists():
    panns_results = json.loads(esc_path.read_text())
    macro = panns_results['Macro']['F1']
    print(f'PANNs CNN14 (fine-tuned): Macro F1 = {macro}')
    all_results['PANNs CNN14 (ESAS)'] = macro
else:
    print('Run notebook 02 first to get PANNs results')

## Model 2 — AST (Audio Spectrogram Transformer)

In [ ]:
try:
    from transformers import ASTFeatureExtractor, ASTModel
    import torch

    AST_CACHE = MODELS_DIR / 'ast_embeddings.npz'

    if AST_CACHE.exists():
        print('Loading cached AST embeddings...')
        d = np.load(AST_CACHE, allow_pickle=True)
        X_a, y_a, folds_a = d['X'], d['y'], d['folds']
    else:
        print('Loading AST model...')
        feature_extractor = ASTFeatureExtractor.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593')
        model = ASTModel.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593')
        model.eval()
        X_a, y_a, folds_a = [], [], []
        for i, path in enumerate(files):
            fn = Path(path).name
            info = meta.get(fn)
            if info is None: continue
            try:
                audio = load_audio(path, sr=16000)
                inputs = feature_extractor(audio, sampling_rate=16000, return_tensors='pt')
                with torch.no_grad():
                    outputs = model(**inputs)
                emb = outputs.pooler_output.squeeze().numpy()
                X_a.append(emb)
                y_a.append(get_priority(info['cat']))
                folds_a.append(info['fold'])
            except Exception as e:
                pass
            if (i+1)%200==0: print(f'  {i+1}/{len(files)}')
        X_a, y_a, folds_a = np.array(X_a), np.array(y_a), np.array(folds_a)
        np.savez(AST_CACHE, X=X_a, y=y_a, folds=folds_a)
        print(f'Cached: {AST_CACHE}')

    print(f'AST embeddings: {X_a.shape}')
    print('Running 5-fold CV...')
    per_class, macro = cross_validate(X_a, y_a, folds_a)
    print(f'AST (fine-tuned): Macro F1 = {macro}')
    print(f'  Per-class: {per_class}')
    all_results['AST (fine-tuned)'] = macro

    # Measure inference time
    import time
    audio = load_audio(files[0], sr=16000)
    inputs = feature_extractor(audio, sampling_rate=16000, return_tensors='pt')
    times = []
    for _ in range(10):
        t0 = time.time()
        with torch.no_grad(): model(**inputs)
        times.append((time.time()-t0)*1000)
    print(f'AST inference: {np.mean(times):.0f} ms (mean over 10 runs)')

except Exception as e:
    print(f'AST error: {e}')
    print('transformers not installed. Install: pip install transformers torch torchaudio')
    print('Skipping AST.')

## Model 3 — CLAP (Contrastive Language-Audio Pretraining)

In [ ]:
try:
    import laion_clap
    import torch

    CLAP_CACHE = MODELS_DIR / 'clap_embeddings.npz'

    if CLAP_CACHE.exists():
        print('Loading cached CLAP embeddings...')
        d = np.load(CLAP_CACHE, allow_pickle=True)
        X_c, y_c, folds_c = d['X'], d['y'], d['folds']
    else:
        print('Loading CLAP model...')
        model = laion_clap.CLAP_Module(enable_fusion=False)
        model.load_ckpt()
        X_c, y_c, folds_c = [], [], []
        for i, path in enumerate(files):
            fn = Path(path).name
            info = meta.get(fn)
            if info is None: continue
            try:
                emb = model.get_audio_embedding_from_filelist([path], use_tensor=False)
                X_c.append(emb[0])
                y_c.append(get_priority(info['cat']))
                folds_c.append(info['fold'])
            except Exception as e:
                pass
            if (i+1)%200==0: print(f'  {i+1}/{len(files)}')
        X_c, y_c, folds_c = np.array(X_c), np.array(y_c), np.array(folds_c)
        np.savez(CLAP_CACHE, X=X_c, y=y_c, folds=folds_c)
        print(f'Cached: {CLAP_CACHE}')

    print(f'CLAP embeddings: {X_c.shape}')
    print('Running 5-fold CV...')
    per_class, macro = cross_validate(X_c, y_c, folds_c)
    print(f'CLAP (fine-tuned): Macro F1 = {macro}')
    print(f'  Per-class: {per_class}')
    all_results['CLAP (fine-tuned)'] = macro

except ImportError:
    print('laion-clap not installed. Install: pip install laion-clap')
    print('Skipping CLAP.')

## Summary Table

In [ ]:
print('='*55)
print('  MODEL COMPARISON — Macro F1 on ESC-50 Hazard Tiers')
print('='*55)
print(f'  {"Model":<35} {"Macro F1":>10}')
print(f'  {"-"*47}')

# Zero-shot baseline: computed by notebook 07 (results/zeroshot_baseline.json)
zs_path = RESULTS_DIR / 'zeroshot_baseline.json'
if zs_path.exists():
    zs_f1 = json.loads(zs_path.read_text())['macro_f1']
else:
    zs_f1 = 0.775  # measured zero-shot macro F1 (run notebook 07 to regenerate)
baselines = {'PANNs CNN14 (no fine-tuning)': zs_f1}
for model, f1 in baselines.items():
    print(f'  {model:<35} {f1:>10.3f}')

for model, f1 in sorted(all_results.items(), key=lambda x: -x[1]):
    marker = ' <-- ESAS' if 'ESAS' in model else ''
    print(f'  {model:<35} {f1:>10.3f}{marker}')

print('='*55)

# Save results
with open(RESULTS_DIR / 'model_comparison.json', 'w') as f:
    import json
    json.dump({**baselines, **all_results}, f, indent=2)
print(f'Saved: {RESULTS_DIR}/model_comparison.json')